# Notebook 12 — Publication-Quality Maps and Figures
**Stage 12 of the Ghana Groundwater–Subsidence Pipeline**

Produces all 10 manuscript figures at 300 dpi from completed pipeline outputs.

| Figure | Content | Primary source |
|--------|---------|----------------|
| Fig 01 | Study area and analytical framework | GADM, coastal buffers |
| Fig 02 | GWSA time series | NB03/04 |
| Fig 03 | Groundwater trend and drought map | NB04 |
| Fig 04 | Climate–groundwater comparison | NB08 / CHIRPS |
| Fig 05 | Urban growth and built-up expansion | NB07 |
| Fig 06 | Urban Groundwater Pressure Index | NB09 |
| Fig 07 | Subsidence proxy and S1 context | NB05 |
| Fig 08 | Bivariate GW–subsidence hazard | NB05+NB03 |
| Fig 09 | CGSRI integrated risk map (main result) | NB10 |
| Fig 10 | District ranking and priority zones | NB10 |

In [ ]:
import os
import pathlib; os.chdir(str(pathlib.Path.cwd().parent) if pathlib.Path.cwd().name == 'notebooks' else str(pathlib.Path.cwd()))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from pathlib import Path
from scipy import stats as sp_stats

ROOT   = Path('.')
BOUND  = ROOT / 'data' / 'raw' / 'boundaries'
GW     = ROOT / 'data' / 'processed' / 'groundwater_anomalies'
SUB    = ROOT / 'data' / 'processed' / 'subsidence'
UG     = ROOT / 'data' / 'processed' / 'urban_growth'
PI     = ROOT / 'data' / 'processed' / 'pressure_indices'
RI     = ROOT / 'data' / 'processed' / 'risk_index'
CLI    = ROOT / 'data' / 'processed' / 'climate_indices'
TABS   = ROOT / 'outputs' / 'tables'
FIGS   = ROOT / 'outputs' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 9,
    'axes.titlesize': 10,
    'axes.labelsize': 9,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
})
DPI = 300

MAP_XLIM = (-3.5, 1.5)
MAP_YLIM = (4.0, 7.5)

# Verified coordinates (WGS84, lon/lat):
#   Accra      -0.187°E  5.603°N  ✓ Accra central
#   Tema       -0.016°E  5.670°N  ✓ Tema port
#   Cape Coast -1.246°E  5.106°N  (was -1.751, 4.934 — below the coast, wrong)
#   Takoradi   -1.752°E  4.894°N  (was -2.020, 5.355 — ~50 km NW of actual city)
#   Ada         0.627°E  5.772°N  (was  0.983, 5.770 — east of Volta mouth, wrong)
#   Keta        0.999°E  5.916°N  (was  0.999, 5.081 — 0.84° too far south, in ocean!)
CITIES = [
    (-0.187,  5.603, 'Accra'),
    (-0.016,  5.670, 'Tema'),
    (-1.246,  5.106, 'Cape Coast'),
    (-1.752,  4.894, 'Takoradi'),
    ( 0.627,  5.772, 'Ada'),
    ( 0.999,  5.916, 'Keta'),
]

RISK_COLORS = {
    'Very low':  '#2166AC',
    'Low':       '#74ADD1',
    'Moderate':  '#FEE08B',
    'High':      '#F46D43',
    'Very high': '#A50026',
}
RISK_ORDER = ['Very low','Low','Moderate','High','Very high']

def map_furniture(ax, xlim=MAP_XLIM, ylim=MAP_YLIM, cities=True):
    ax.set_xlim(xlim); ax.set_ylim(ylim)
    ax.set_xlabel('Longitude (°E)', fontsize=8)
    ax.set_ylabel('Latitude (°N)', fontsize=8)
    for x in np.arange(np.ceil(xlim[0]), xlim[1]+0.5, 1):
        ax.axvline(x, color='#CCCCCC', lw=0.3, zorder=0)
    for y in np.arange(np.ceil(ylim[0]), ylim[1]+0.5, 1):
        ax.axhline(y, color='#CCCCCC', lw=0.3, zorder=0)
    if cities:
        for lon, lat, nm in CITIES:
            if xlim[0] <= lon <= xlim[1] and ylim[0] <= lat <= ylim[1]:
                ax.plot(lon, lat, 'k^', ms=4, zorder=5)
                ax.annotate(nm, (lon, lat), xytext=(4,3),
                            textcoords='offset points', fontsize=6, zorder=6)

def north_arrow(ax, x=0.96, y=0.94):
    ax.annotate('N', xy=(x, y), xycoords='axes fraction',
                fontsize=9, fontweight='bold', ha='center', va='bottom')
    ax.annotate('', xy=(x, y), xytext=(x, y-0.06), xycoords='axes fraction',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

# Load boundaries once
ghana     = gpd.read_file(BOUND / 'ghana_boundary.gpkg').to_crs('EPSG:4326')
try:
    coast50   = gpd.read_file(BOUND / 'study_area_coastal_50km.gpkg').to_crs('EPSG:4326')
except Exception: coast50 = None
try:
    districts = gpd.read_file(BOUND / 'coastal_districts.gpkg').to_crs('EPSG:4326')
except Exception: districts = None
try:
    grid10    = gpd.read_file(BOUND / 'analysis_grid_10km.gpkg').to_crs('EPSG:4326')
except Exception: grid10 = None

print('Setup OK')


## Fig 01 — Study Area

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 5.5))

# (a) Ghana context
ax = axes[0]
ghana.plot(ax=ax, facecolor='#F5F0E8', edgecolor='black', lw=0.8)
if coast50 is not None:
    coast50.plot(ax=ax, facecolor='#C6EFCE', edgecolor='#538135', lw=0.8, alpha=0.55)
# Verified coords for panel (a) star markers (lon, lat):
#   Accra    -0.187°E  5.603°N  ✓
#   Kumasi   -1.624°E  6.688°N  ✓
#   Takoradi -1.752°E  4.894°N  (Sekondi-Takoradi port)
for lon, lat, nm in [(-0.187, 5.603, 'Accra'),
                     (-1.624, 6.688, 'Kumasi'),
                     (-1.752, 4.894, 'Takoradi')]:
    ax.plot(lon, lat, 'r*', ms=7, zorder=5)
    ax.annotate(nm, (lon, lat), xytext=(4,4), textcoords='offset points', fontsize=8)
ax.set_xlim(-3.8, 1.6); ax.set_ylim(4.5, 11.3)
ax.set_title('(a) Ghana — study area location', fontsize=9)
ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude (°N)')

# Africa inset — highlight shows COASTAL STRIP only (4.5–7.0°N)
axins = ax.inset_axes([0.02, 0.55, 0.3, 0.42])
axins.set_facecolor('#D0E8F0')
axins.set_xlim(-20, 55); axins.set_ylim(-35, 40)
# Rectangle(xy_lower_left, width, height): height=2.5 spans 4.5°N to 7.0°N
axins.add_patch(Rectangle((-3.8, 4.5), 5.4, 2.5,
                facecolor='#CC4444', alpha=0.70, edgecolor='red', lw=1.2))
axins.text(10, -20, 'Africa', fontsize=5, color='navy')
axins.tick_params(left=False, bottom=False, labelleft=False, labelbottom=False)

# (b) Coastal detail
ax = axes[1]
if districts is not None:
    districts.plot(ax=ax, facecolor='#E8EEF4', edgecolor='#888888', lw=0.5)
else:
    ghana.plot(ax=ax, facecolor='#E8EEF4', edgecolor='black', lw=0.8)
if coast50 is not None:
    coast50.boundary.plot(ax=ax, edgecolor='#538135', lw=1.5, linestyle='--',
                          label='50 km coastal buffer')
if grid10 is not None:
    grid10.plot(ax=ax, facecolor='none', edgecolor='#BBBBBB', lw=0.2, alpha=0.6)
map_furniture(ax, cities=True)
north_arrow(ax)
ax.set_title('(b) Coastal study area and 10 km analysis grid', fontsize=9)
ax.legend(loc='lower right', fontsize=7)
# Corridor / area text labels — verified positions (lon, lat)
for lon, lat, nm in [(-0.10,  5.70, 'Accra–Tema corridor'),
                     (-1.25,  5.10, 'Cape Coast'),
                     (-1.75,  5.05, 'Sekondi-\nTakoradi'),
                     ( 0.80,  6.05, 'Ada coast')]:
    if MAP_XLIM[0] <= lon <= MAP_XLIM[1] and MAP_YLIM[0] <= lat <= MAP_YLIM[1]:
        ax.text(lon, lat, nm, fontsize=5.5, color='#2c5f2e', ha='center',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.75))

plt.tight_layout()
fig.savefig(FIGS / 'fig01_study_area.png', dpi=DPI)
plt.close()
print('Saved fig01_study_area.png')
print('  Africa inset: red box now spans 4.5–7.0°N (coastal strip, h=2.5°)')


## Fig 02 — GWSA Time Series

In [ ]:
TS_DIR   = ROOT / 'data' / 'interim' / 'time_series'
gwsa_csv = TS_DIR / 'gwsa_monthly.csv'
if not gwsa_csv.exists():
    gwsa_csv = GW / 'gwsa_monthly.csv'

df_gwsa_raw = pd.read_csv(gwsa_csv)
# Handle either 'date' or 'time' as the datetime column
time_col = 'date' if 'date' in df_gwsa_raw.columns else 'time'
df_gwsa_raw[time_col] = pd.to_datetime(df_gwsa_raw[time_col])
df_gwsa  = df_gwsa_raw.set_index(time_col)
gwsa_ts  = df_gwsa['gwsa_mm']

t_num = (gwsa_ts.index - gwsa_ts.index[0]).days.values.astype(float)
slope, intercept, r, p, se = sp_stats.linregress(t_num, gwsa_ts.ffill().values)
trend_yr   = slope * 365.25
trend_line = slope * t_num + intercept

gwsa_ann = gwsa_ts.resample('YE').mean()
gwsa_clim_mean = gwsa_ts.groupby(gwsa_ts.index.month).mean()
gwsa_clim_std  = gwsa_ts.groupby(gwsa_ts.index.month).std()

fig = plt.figure(figsize=(11, 7))
gs  = GridSpec(2, 2, figure=fig, hspace=0.45, wspace=0.35)

ax = fig.add_subplot(gs[0, :])
ax.fill_between(gwsa_ts.index, np.where(gwsa_ts < 0, gwsa_ts, 0), 0,
                color='#CC4444', alpha=0.4, label='Deficit')
ax.fill_between(gwsa_ts.index, np.where(gwsa_ts >= 0, gwsa_ts, 0), 0,
                color='#4477AA', alpha=0.4, label='Surplus')
ax.plot(gwsa_ts.index, gwsa_ts.rolling(12, center=True).mean(),
        color='navy', lw=1.6, label='12-month mean')
ax.plot(gwsa_ts.index, trend_line, color='#CC4444', lw=1.2, ls='--',
        label=f'Trend: {trend_yr:+.1f} mm yr⁻¹ (p={p:.3f})')
ax.axhline(0, color='k', lw=0.7)
ax.axvspan(pd.Timestamp('2017-06-01'), pd.Timestamp('2018-06-01'),
           color='gray', alpha=0.14, label='GRACE mission gap')
ax.set_ylabel('GWSA (mm)')
ax.set_title('(a) Monthly GWSA with linear trend and 12-month running mean', fontsize=9)
ax.legend(ncol=3, fontsize=7, loc='upper left')

ax2 = fig.add_subplot(gs[1, 0])
bar_c = ['#CC4444' if v < 0 else '#4477AA' for v in gwsa_ann.values]
ax2.bar(gwsa_ann.index.year, gwsa_ann.values, color=bar_c, width=0.7, edgecolor='white')
ax2.axhline(0, color='k', lw=0.7)
ax2.set_ylabel('Annual mean GWSA (mm)')
ax2.set_title('(b) Annual mean GWSA', fontsize=9)
ax2.tick_params(axis='x', rotation=45)

ax3 = fig.add_subplot(gs[1, 1])
months = np.arange(1, 13)
ax3.bar(months, gwsa_clim_mean.values, color='steelblue', width=0.7, alpha=0.7)
ax3.errorbar(months, gwsa_clim_mean.values, yerr=gwsa_clim_std.values,
             fmt='none', ecolor='navy', capsize=3, lw=0.8)
ax3.axhline(0, color='k', lw=0.7)
ax3.set_xticks(months)
ax3.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax3.set_ylabel('GWSA (mm)')
ax3.set_title('(c) Mean seasonal cycle (±1 SD)', fontsize=9)

plt.savefig(FIGS / 'fig02_gwsa_timeseries.png', dpi=DPI)
plt.close()
print('Saved fig02_gwsa_timeseries.png')

## Fig 03 — Groundwater Trend and Drought Map

In [ ]:
ds_trend   = xr.open_dataset(GW / 'gwsa_trend_pixelwise.nc')
ds_drought = xr.open_dataset(GW / 'gdsi_drought_metrics_grid.nc')

slope  = ds_trend['sens_slope_mm_yr'].values
pval   = ds_trend['mk_pvalue'].values
dfreq  = ds_drought['drought_freq_pct'].values
lats_t = ds_trend.lat.values
lons_t = ds_trend.lon.values

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

ax = axes[0]
vabs = np.nanmax(np.abs(np.nanpercentile(slope, [5, 95])))
im = ax.pcolormesh(lons_t, lats_t, slope, cmap='RdBu',
                   vmin=-vabs, vmax=vabs, shading='auto')
# Stipple significant
sig_mask = pval < 0.1
li, lj = np.where(sig_mask)
ax.plot(lons_t[lj], lats_t[li], '.', color='k', ms=1.5, alpha=0.45)
bnd = districts if districts is not None else ghana
bnd.boundary.plot(ax=ax, edgecolor='#444444', lw=0.4)
plt.colorbar(im, ax=ax, shrink=0.85, label="Sen's slope (mm yr⁻¹)")
map_furniture(ax)
north_arrow(ax)
ax.set_title('(a) GWSA linear trend (MK p < 0.1 stippled)', fontsize=9)

ax = axes[1]
im2 = ax.pcolormesh(lons_t, lats_t, dfreq, cmap='YlOrRd',
                    vmin=0, vmax=60, shading='auto')
bnd.boundary.plot(ax=ax, edgecolor='#444444', lw=0.4)
plt.colorbar(im2, ax=ax, shrink=0.85, label='% months SGWA < −0.5')
map_furniture(ax)
north_arrow(ax)
ax.set_title('(b) Groundwater drought frequency\n(SGWA < −0.5)', fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / 'fig03_gwsa_trend_drought_map.png', dpi=DPI)
plt.close()
print('Saved fig03_gwsa_trend_drought_map.png')

## Fig 04 — Climate–Groundwater Comparison

In [ ]:
tc_file   = CLI / 'terraclimate_coastal_timeseries.csv'
rain_file = CLI / 'rainfall_anomaly_monthly.csv'

fig, axes = plt.subplots(3, 1, figsize=(11, 8.5), sharex=True)

# (a) GWSA
ax = axes[0]
ax.fill_between(gwsa_ts.index, np.where(gwsa_ts < 0, gwsa_ts, 0), 0,
                color='#CC4444', alpha=0.4, label='Deficit')
ax.fill_between(gwsa_ts.index, np.where(gwsa_ts >= 0, gwsa_ts, 0), 0,
                color='#4477AA', alpha=0.4, label='Surplus')
ax.plot(gwsa_ts.index, gwsa_ts.rolling(12, center=True).mean(),
        color='navy', lw=1.5)
ax.axhline(0, color='k', lw=0.7, ls='--')
ax.axvspan(pd.Timestamp('2017-06-01'), pd.Timestamp('2018-06-01'),
           color='gray', alpha=0.12, label='GRACE gap')
ax.set_ylabel('GWSA (mm)')
ax.set_title('(a) Groundwater storage anomaly (GRACE–GLDAS)', fontsize=9)
ax.legend(fontsize=7, ncol=3)

# (b) PDSI (if NB08 run) else rainfall anomaly
ax = axes[1]
if tc_file.exists():
    df_tc   = pd.read_csv(tc_file, parse_dates=['date'])
    pdsi_ts = df_tc.set_index('date')['pdsi']
    ax.fill_between(pdsi_ts.index, np.where(pdsi_ts < 0, pdsi_ts, 0), 0,
                    color='#FF6B35', alpha=0.4, label='Drought (PDSI < 0)')
    ax.fill_between(pdsi_ts.index, np.where(pdsi_ts >= 0, pdsi_ts, 0), 0,
                    color='#4682B4', alpha=0.4, label='Wet')
    ax.plot(pdsi_ts.index, pdsi_ts.rolling(3, center=True).mean(),
            color='darkred', lw=1.2)
    ax.axhline(-1.0, color='orange', lw=0.7, ls=':', label='Mild drought (−1)')
    ax.set_ylim(-6, 6)
    ax.set_ylabel('PDSI')
    ax.set_title('(b) Palmer Drought Severity Index (TerraClimate)', fontsize=9)
elif rain_file.exists():
    df_r = pd.read_csv(rain_file, parse_dates=['date'])
    ra   = df_r.set_index('date')['precip_anom_mm']
    ax.bar(ra.index, ra.values, width=25,
           color=['#CC4444' if v < 0 else '#4477AA' for v in ra.values], alpha=0.6)
    ax.axhline(0, color='k', lw=0.7)
    ax.set_ylabel('Rainfall anomaly (mm)')
    ax.set_title('(b) CHIRPS monthly rainfall anomaly (TerraClimate pending)', fontsize=9)
else:
    ax.text(0.5, 0.5, 'Run Notebook 08 for TerraClimate PDSI',
            ha='center', va='center', transform=ax.transAxes, color='gray')
    ax.set_title('(b) Palmer Drought Severity Index (pending)', fontsize=9)
ax.legend(fontsize=7)

# (c) CHIRPS monthly rainfall
ax = axes[2]
if rain_file.exists():
    df_r2 = pd.read_csv(rain_file, parse_dates=['date'])
    prec  = df_r2.set_index('date')['precip_mm']
    ax.bar(prec.index, prec.values, width=25, color='steelblue', alpha=0.55, label='Monthly')
    ax.plot(prec.index, prec.rolling(12, center=True).mean(),
            color='navy', lw=1.4, label='12-month mean')
    ax.set_ylabel('Precipitation (mm month⁻¹)')
    ax.legend(fontsize=7)
ax.set_title('(c) CHIRPS monthly precipitation', fontsize=9)
ax.set_xlabel('Year')

plt.tight_layout()
plt.savefig(FIGS / 'fig04_climate_groundwater.png', dpi=DPI)
plt.close()
print('Saved fig04_climate_groundwater.png')

## Fig 05 — Urban Growth

In [ ]:
gdf_ug  = gpd.read_file(UG / 'builtup_grid10km.gpkg').to_crs('EPSG:4326')
df_buts = pd.read_csv(UG / 'builtup_area_timeseries.csv')

gdf_ug['density_frac'] = gdf_ug['builtup_m2_2020'] / (gdf_ug['area_km2'] * 1e6)
gdf_ug['growth_pct']   = gdf_ug['builtup_growth_pct'].clip(0, 400)

fig, axes = plt.subplots(1, 3, figsize=(13, 5.5))

ax = axes[0]
vmax_d = gdf_ug['density_frac'].quantile(0.97)
gdf_ug.plot(column='density_frac', ax=ax, cmap='Oranges', vmin=0, vmax=vmax_d,
            edgecolor='none', legend=True,
            legend_kwds={'label': 'Built-up fraction (2020)', 'shrink': 0.8})
ghana.boundary.plot(ax=ax, edgecolor='#555555', lw=0.5)
map_furniture(ax)
north_arrow(ax)
ax.set_title('(a) Built-up density 2020', fontsize=9)

ax = axes[1]
vmax_g = gdf_ug['growth_pct'].quantile(0.95)
gdf_ug.plot(column='growth_pct', ax=ax, cmap='RdYlGn_r', vmin=0, vmax=vmax_g,
            edgecolor='none', missing_kwds={'color': 'lightgray'}, legend=True,
            legend_kwds={'label': 'Built-up growth 2000–2025 (%)', 'shrink': 0.8})
ghana.boundary.plot(ax=ax, edgecolor='#555555', lw=0.5)
map_furniture(ax)
north_arrow(ax)
ax.set_title('(b) Built-up growth rate 2000–2025 (%)', fontsize=9)

ax = axes[2]
print('Built-up timeseries columns:', df_buts.columns.tolist())
yr_data = {}
for yr in [2000, 2010, 2015, 2020, 2025]:
    col = [c for c in df_buts.columns if str(yr) in str(c)]
    if col:
        total_m2 = df_buts[col[0]].sum()
        yr_data[yr] = total_m2 / 1e6   # → km²
if not yr_data:
    # Try row-oriented CSV — prefer built_area_km2 if present
    if 'year' in df_buts.columns:
        if 'built_area_km2' in df_buts.columns:
            area_col = 'built_area_km2'
        elif 'built_area_m2' in df_buts.columns:
            area_col = 'built_area_m2'
        else:
            area_col = next((c for c in df_buts.columns if 'km2' in c.lower()), None) or \
                        next((c for c in df_buts.columns if 'area' in c.lower()), None)
        if area_col:
            scale = 1e-6 if 'm2' in area_col.lower() and 'km2' not in area_col.lower() else 1.0
            for _, row in df_buts.iterrows():
                yr_data[int(row['year'])] = float(row[area_col]) * scale

if yr_data:
    yrs  = sorted(yr_data.keys())
    vals = [yr_data[y] for y in yrs]
    bars = ax.bar(yrs, vals, color='#CC6633', width=3.5, alpha=0.75)
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, v+5, f'{v:.0f}', ha='center', fontsize=7.5)
    ax.set_ylabel('Built-up area (km²)')
    ax.set_title('(c) Built-up area 2000–2025', fontsize=9)
    ax.set_xlabel('Year')
    growth = (vals[-1] - vals[0]) / vals[0] * 100 if vals[0] > 0 else 0
    ax.text(0.03, 0.95, f'+{growth:.0f}% in 25 years',
            transform=ax.transAxes, fontsize=8, va='top',
            bbox=dict(boxstyle='round', fc='#FFF3CD', alpha=0.9))
else:
    ax.text(0.5, 0.5, 'See builtup_area_timeseries.csv',
            ha='center', va='center', transform=ax.transAxes, color='gray')
    ax.set_title('(c) Built-up expansion', fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / 'fig05_urban_growth.png', dpi=DPI)
plt.close()
print('Saved fig05_urban_growth.png')

## Fig 06 — UGPI

In [ ]:
gdf_ugpi = gpd.read_file(PI / 'ugpi_grid10km.gpkg').to_crs('EPSG:4326')
ugpi_col = 'ugpi_equal' if 'ugpi_equal' in gdf_ugpi.columns else \
           next(c for c in gdf_ugpi.columns if 'ugpi' in c.lower())
n_cols = [c for c in gdf_ugpi.columns if c.startswith('n_')]

fig, axes = plt.subplots(1, 2, figsize=(11, 5.5))

ax = axes[0]
gdf_ugpi.plot(column=ugpi_col, ax=ax, cmap='RdYlGn_r',
              vmin=0, vmax=gdf_ugpi[ugpi_col].quantile(0.98),
              edgecolor='none', legend=True,
              legend_kwds={'label': 'UGPI (0–1, high = more pressure)', 'shrink': 0.8})
bnd = districts if districts is not None else ghana
bnd.boundary.plot(ax=ax, edgecolor='#444444', lw=0.4)
map_furniture(ax)
north_arrow(ax)
ax.set_title('(a) Urban Groundwater Pressure Index', fontsize=9)

ax = axes[1]
if len(n_cols) >= 2:
    ind_means = gdf_ugpi[n_cols].mean().sort_values()
    short_lbl = [c.replace('n_','') for c in ind_means.index]
    colors_b  = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(ind_means)))
    ax.barh(range(len(ind_means)), ind_means.values, color=colors_b)
    ax.set_yticks(range(len(ind_means)))
    ax.set_yticklabels(short_lbl)
    ax.axvline(0.5, color='k', lw=0.7, ls='--', alpha=0.4)
    for i, v in enumerate(ind_means.values):
        ax.text(v+0.01, i, f'{v:.3f}', va='center', fontsize=7)
    ax.set_xlabel('Mean normalised indicator value')
    ax.set_title('(b) Indicator contributions (study-area mean)', fontsize=9)
else:
    top20 = gdf_ugpi.nlargest(20, ugpi_col)
    ax.barh(range(20), top20[ugpi_col].values[::-1], color='#CC4444', alpha=0.7)
    ax.set_yticks(range(20))
    ax.set_yticklabels([str(c) for c in top20['cell_id'].values[::-1]], fontsize=7)
    ax.set_xlabel('UGPI')
    ax.set_title('(b) Top-20 high-pressure cells', fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / 'fig06_ugpi_map.png', dpi=DPI)
plt.close()
print('Saved fig06_ugpi_map.png')

## Fig 07 — Subsidence Proxy and S1 Inventory

In [ ]:
# ── Fig 07: Subsidence Proxy and ERA5-corrected S1 InSAR Velocity ───────────
# ERA5 tropospheric correction (pyAPS3 + cfgrib) applied to MintPy SBAS timeseries.
# Post-correction, temporal coherence mask (≥0.7) passes 3,222,917 pixels (WGS84: 3.4M)
# covering 5.0–7.4°N — the full scene footprint.
# IMPORTANT CAVEAT: pixel distribution is ~300 k/0.2° band (near-uniform) across the
# scene. In tropical coastal zones (5.0–7.0°N) dense vegetation typically limits
# interferometric coherence; the uniform distribution suggests possible SBAS noise
# fitting rather than real coherent ground signal. Use 7.0–7.4°N (inland savanna)
# as the higher-confidence zone. Coastal velocities are shown for completeness but
# should be treated as exploratory pending PS-InSAR or GPS validation.
from rasterio.warp import reproject as _rp, Resampling, calculate_default_transform as _cdt
import rasterio

INSAR_NOTE = ('MintPy SBAS: 159 HyP3-ISCE pairs, 2016–2024\n'
              'Track 147 ascending  |  ERA5 tropospheric correction applied\n'
              'Valid pixels (coherence ≥0.7): 3,222,917\n'
              'Post-correction extent: 5.0–7.4°N\n'
              'Higher confidence: 7.0–7.4°N (inland savanna)\n'
              'Coastal zone (5.0–7.0°N): uniform pixel distribution —\n'
              'possible SBAS noise fitting; interpret with caution.\n'
              'Velocity range (ERA5-corrected): −60 to +15 mm/yr')

# ── helper: reproject raster to WGS84 in memory ──────────────────────────────
def _to_wgs84(src_path):
    with rasterio.open(src_path) as src:
        dst_crs = 'EPSG:4326'
        tr, w, h = _cdt(src.crs, dst_crs, src.width, src.height, *src.bounds)
        raw = src.read(1).astype(np.float32)
        nd  = src.nodata
        if nd is not None: raw[raw == nd] = np.nan
        dst = np.full((h, w), np.nan, dtype=np.float32)
        _rp(source=raw, destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=tr, dst_crs=dst_crs,
            resampling=Resampling.bilinear, src_nodata=np.nan, dst_nodata=np.nan)
    ext = [tr.c, tr.c + tr.a*w, tr.f + tr.e*h, tr.f]   # [W, E, S, N]
    return dst, ext

vel_era5_path = SUB / 'mintpy_velocity_los_era5.tif'

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

# ─── panel (a): CGSRI subsidence proxy ───────────────────────────────────────
ax = axes[0]
try:
    gdf_sub = gpd.read_file(RI / 'cgsri_grid10km.gpkg').to_crs('EPSG:4326')
    vmax_s  = gdf_sub['sub_hazard'].quantile(0.97)
    gdf_sub.plot(column='sub_hazard', ax=ax, cmap='RdYlGn_r',
                 vmin=0, vmax=vmax_s, edgecolor='none', legend=True,
                 legend_kwds={'label': 'Subsidence proxy score (0–1)', 'shrink': 0.85})
except Exception as e_sub:
    ax.text(0.5, 0.5, f'sub_hazard unavailable\n{e_sub}', ha='center', va='center',
            transform=ax.transAxes, color='gray', fontsize=8)
bnd = districts if districts is not None else ghana
bnd.boundary.plot(ax=ax, edgecolor='#555555', lw=0.35, zorder=3)
ax.set_xlim(-3.8, 1.6); ax.set_ylim(4.4, 7.2)
ax.set_xlabel('Longitude (°E)', fontsize=8); ax.set_ylabel('Latitude (°N)', fontsize=8)
ax.set_title('(a) Composite subsidence proxy\n'
             '(elevation + building density + groundwater demand)', fontsize=9)
ax.text(0.99, 0.01, INSAR_NOTE, transform=ax.transAxes, fontsize=5.0,
        va='bottom', ha='right', color='#333333',
        bbox=dict(boxstyle='round', fc='white', alpha=0.88, edgecolor='#AAAAAA', lw=0.5))

# ─── panel (b): ERA5-corrected LOS velocity map ───────────────────────────────
ax = axes[1]
try:
    vel_wgs84, ext = _to_wgs84(vel_era5_path)
    vmin_v, vmax_v = -40, 20
    im = ax.imshow(vel_wgs84, cmap='RdBu', vmin=vmin_v, vmax=vmax_v,
                   extent=[ext[0], ext[1], ext[2], ext[3]],
                   origin='upper', aspect='auto', interpolation='nearest', zorder=2)
    cbar = plt.colorbar(im, ax=ax, shrink=0.85,
                        label='LOS velocity (mm/yr, +ve = uplift)')
    cbar.ax.tick_params(labelsize=7)
except Exception as e_v:
    ax.text(0.5, 0.5, f'ERA5 velocity unavailable\n{e_v}',
            ha='center', va='center', transform=ax.transAxes, color='gray', fontsize=8)

bnd.boundary.plot(ax=ax, edgecolor='#333333', lw=0.35, zorder=3)
ax.axhline(7.0, color='gold', lw=1.2, ls='--', alpha=0.85, zorder=5)
ax.text(-0.1, 7.06, 'Higher confidence (inland savanna ≥7.0°N)',
        fontsize=5.5, color='goldenrod', ha='center', va='bottom',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.78, edgecolor='none'))
ax.text(-0.1, 6.88, 'Caution: coastal zone\n(SBAS noise possible)',
        fontsize=5.5, color='#994400', ha='center', va='top',
        bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.78, edgecolor='none'))
ax.set_xlim(-3.8, 1.6); ax.set_ylim(4.4, 7.5)
ax.set_xlabel('Longitude (°E)', fontsize=8); ax.set_ylabel('Latitude (°N)', fontsize=8)
ax.set_title('(b) ERA5-corrected S1 LOS velocity (mm/yr)\n'
             '(coverage: 5.0–7.4°N post-correction; coherence ≥0.7)', fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / 'fig07_subsidence_s1.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved fig07_subsidence_s1.png')


## Fig 08 — Bivariate GW–Subsidence Map

In [ ]:
gdf_cs = gpd.read_file(RI / 'cgsri_grid10km.gpkg').to_crs('EPSG:4326')

# Use ugpi_equal (Urban Groundwater Pressure Index) instead of gw_hazard:
# gw_hazard ≈ 0 for virtually all cells (GRACE resolution limitation),
# producing a degenerate bivariate map.  UGPI has meaningful spatial variation
# (min=0.025, max=0.346) reflecting building density, drought, and demand pressure.
ugpi_med = gdf_cs['ugpi_equal'].median()
sub_med  = gdf_cs['sub_hazard'].median()
gdf_cs['bivar'] = gdf_cs.apply(
    lambda r: ('H' if r['ugpi_equal'] >= ugpi_med else 'L') + '/' +
              ('H' if r['sub_hazard']  >= sub_med  else 'L'),
    axis=1
)
bivar_colors = {'L/L':'#F7F7F7', 'H/L':'#74ADD1', 'L/H':'#F46D43', 'H/H':'#A50026'}
bivar_labels = {
    'L/L': 'Low UGPI + low subsidence proxy',
    'H/L': 'High UGPI + low subsidence proxy',
    'L/H': 'Low UGPI + high subsidence proxy',
    'H/H': 'HIGH UGPI + HIGH subsidence proxy',
}

fig, axes = plt.subplots(1, 2, figsize=(12, 5.5))

ax = axes[0]
# Background for cells with no class
gdf_cs.plot(ax=ax, facecolor='#F7F7F7', edgecolor='none')
for cls, color in bivar_colors.items():
    sub = gdf_cs[gdf_cs['bivar'] == cls]
    if len(sub) > 0:
        sub.plot(ax=ax, facecolor=color, edgecolor='none')
bnd = districts if districts is not None else ghana
bnd.boundary.plot(ax=ax, edgecolor='#555555', lw=0.4)
handles = [mpatches.Patch(fc=c,
           label=f'{bivar_labels[k]} (n={len(gdf_cs[gdf_cs["bivar"]==k])})')
           for k, c in bivar_colors.items()]
ax.legend(handles=handles, fontsize=6.5, loc='lower right')
map_furniture(ax)
north_arrow(ax)
ax.set_title('(a) Bivariate UGPI × subsidence proxy (median split)', fontsize=9)

ax = axes[1]
sc = ax.scatter(gdf_cs['ugpi_equal'], gdf_cs['sub_hazard'],
                c=gdf_cs['cgsri'], cmap='RdYlGn_r', s=8, alpha=0.45, linewidths=0)
plt.colorbar(sc, ax=ax, label='CGSRI composite score', shrink=0.85)
ax.axvline(ugpi_med, color='navy',   lw=0.9, ls='--', alpha=0.6,
           label=f'UGPI median ({ugpi_med:.3f})')
ax.axhline(sub_med,  color='orange', lw=0.9, ls='--', alpha=0.6,
           label=f'Sub median ({sub_med:.3f})')
ax.set_xlabel('UGPI — Urban Groundwater Pressure Index (equal weights, normalised)')
ax.set_ylabel('Subsidence proxy hazard (normalised)')
ax.legend(fontsize=7)
ax.set_title('(b) Scatter: UGPI vs subsidence proxy\ncoloured by CGSRI composite', fontsize=9)

# Annotation explaining UGPI on x-axis
ax.text(0.01, 0.99,
        'UGPI components: drought frequency, piped-water access gap,\n'
        'built-up growth rate, building density, population density, GW anomaly',
        transform=ax.transAxes, fontsize=5.5, va='top', ha='left', color='#333333',
        bbox=dict(boxstyle='round', fc='white', alpha=0.80, edgecolor='none'))

plt.tight_layout()
plt.savefig(FIGS / 'fig08_bivariate_gw_subsidence.png', dpi=DPI)
plt.close()
print('Saved fig08_bivariate_gw_subsidence.png')
print(f'  x-axis: ugpi_equal (median={ugpi_med:.3f})  y-axis: sub_hazard (median={sub_med:.3f})')
bivar_counts = gdf_cs['bivar'].value_counts()
print('  Class counts:', bivar_counts.to_dict())


## Fig 09 — CGSRI Risk Map (Main Result)

In [ ]:
fig = plt.figure(figsize=(13, 7))
gs  = GridSpec(1, 3, figure=fig, wspace=0.08, width_ratios=[2.8, 0.12, 1.1])

ax_map = fig.add_subplot(gs[0])
# Plot all cells as background
gdf_cs.plot(ax=ax_map, facecolor='#EEEEEE', edgecolor='none')
# Overlay risk classes
for cls in RISK_ORDER:
    sub = gdf_cs[gdf_cs['cgsri_class'] == cls]
    if len(sub) > 0:
        sub.plot(ax=ax_map, fc=RISK_COLORS[cls], edgecolor='none')
# Hatch very-high cells
vh = gdf_cs[gdf_cs['cgsri_class'] == 'Very high']
if len(vh) > 0:
    vh.plot(ax=ax_map, fc='none', edgecolor='black', lw=0.5, hatch='///')
bnd = districts if districts is not None else ghana
bnd.boundary.plot(ax=ax_map, edgecolor='#555555', lw=0.4)
handles = [
    mpatches.Patch(fc=RISK_COLORS[cls],
                   label=f'{cls} (n={len(gdf_cs[gdf_cs["cgsri_class"]==cls])})')
    for cls in RISK_ORDER
]
ax_map.legend(handles=handles, fontsize=7.5, loc='lower right', framealpha=0.9)
map_furniture(ax_map)
north_arrow(ax_map)
ax_map.set_title('CGSRI risk classification (quintiles)\n(hatching = Very High)', fontsize=9)

ax_cb = fig.add_subplot(gs[1])
cmap_r = mcolors.LinearSegmentedColormap.from_list(
    'risk', [RISK_COLORS[c] for c in RISK_ORDER])
norm_r = mcolors.Normalize(vmin=gdf_cs['cgsri'].min(), vmax=gdf_cs['cgsri'].max())
sm = plt.cm.ScalarMappable(cmap=cmap_r, norm=norm_r)
sm.set_array([])
plt.colorbar(sm, cax=ax_cb, label='CGSRI value')

ax_hist = fig.add_subplot(gs[2])
bins = np.linspace(gdf_cs['cgsri'].min(), gdf_cs['cgsri'].max(), 25)
for cls in RISK_ORDER:
    vals = gdf_cs[gdf_cs['cgsri_class'] == cls]['cgsri'].values
    if len(vals) > 0:
        ax_hist.hist(vals, bins=bins, fc=RISK_COLORS[cls], label=cls, alpha=0.88)
ax_hist.set_xlabel('CGSRI value')
ax_hist.set_ylabel('N grid cells')
ax_hist.set_title('CGSRI distribution\nby risk class', fontsize=8)
ax_hist.legend(fontsize=6.5)

# Summary stats box
hi_pop = gdf_cs[gdf_cs['cgsri_class'].isin(['Very high','High'])]['population_2020'].sum()
ax_hist.text(0.03, 0.38,
             f'Very High + High:\n{hi_pop/1e6:.1f}M people',
             transform=ax_hist.transAxes, fontsize=7.5,
             bbox=dict(boxstyle='round', fc='#FFF3CD', alpha=0.9))

plt.savefig(FIGS / 'fig09_cgsri_risk_map.png', dpi=DPI)
plt.close()
print('Saved fig09_cgsri_risk_map.png')

## Fig 10 — District Ranking and Priority Zones

In [ ]:
df_dist = pd.read_csv(TABS / 'cgsri_district_ranking.csv')
df_pz   = pd.read_csv(TABS / 'cgsri_priority_zones.csv')

name_col = next((c for c in df_dist.columns
                 if any(t in c.lower() for t in ('district','region','name'))),
                df_dist.columns[0])
val_col  = next((c for c in df_dist.columns
                 if any(t in c.lower() for t in ('cgsri','mean','score','risk'))),
                df_dist.columns[1])

# Load all 16 GADM regions to identify any not covered by the analysis grid
try:
    gadm = gpd.read_file(BOUND / 'gadm41_GHA.gpkg', layer='ADM_ADM_1')
    all_regions = set(gadm['NAME_1'].tolist())
    covered = set(df_dist[name_col].tolist())
    missing_regions = sorted(all_regions - covered)
except Exception:
    missing_regions = []

fig, axes = plt.subplots(1, 2, figsize=(13, 6.5))

# (a) Regional ranking
ax = axes[0]
n_shown = len(df_dist)
df_sorted = df_dist.sort_values(val_col, ascending=True)   # ascending so highest is at top
q80 = df_dist[val_col].quantile(0.8)
q60 = df_dist[val_col].quantile(0.6)
bar_c = [RISK_COLORS['Very high'] if v >= q80 else
         RISK_COLORS['High']      if v >= q60 else
         RISK_COLORS['Moderate']  for v in df_sorted[val_col].values]
ax.barh(range(n_shown), df_sorted[val_col].values, color=bar_c,
        edgecolor='white', height=0.7)
ax.set_yticks(range(n_shown))
ax.set_yticklabels(df_sorted[name_col].values, fontsize=8)
ax.axvline(df_dist[val_col].mean(), color='k', lw=0.9, ls='--', alpha=0.5,
           label=f'Mean: {df_dist[val_col].mean():.3f}')
ax.set_xlabel('Mean CGSRI')
ax.set_title(f'(a) All {n_shown} regions in analysis grid, ranked by mean CGSRI', fontsize=9)
ax.legend(fontsize=7)
# Annotate missing regions
if missing_regions:
    ax.text(0.98, 0.02,
            f'Not in grid: {", ".join(missing_regions)}\n'
            f'(no intersection with 10 km analysis grid)',
            transform=ax.transAxes, ha='right', va='bottom', fontsize=6.5,
            color='#555555', style='italic',
            bbox=dict(boxstyle='round', fc='#F8F8F8', alpha=0.85))

# (b) Priority zones scatter
ax = axes[1]
pz_cgsri = next((c for c in df_pz.columns if 'cgsri' in c.lower() and 'class' not in c.lower()), None)
pz_pop   = next((c for c in df_pz.columns if 'pop' in c.lower()), None)

if pz_cgsri and pz_pop:
    n_pz = len(df_pz)
    sc = ax.scatter(df_pz[pz_cgsri], df_pz[pz_pop] / 1000,
                    c=df_pz[pz_cgsri], cmap='RdYlGn_r',
                    vmin=df_pz[pz_cgsri].min(), vmax=df_pz[pz_cgsri].max(),
                    s=80, alpha=0.8, edgecolors='black', linewidths=0.4, zorder=3)
    plt.colorbar(sc, ax=ax, label='CGSRI value', shrink=0.85)
    ax.set_xlabel('CGSRI value')
    ax.set_ylabel('Cell population (thousands)')
    ax.set_title(f'(b) Priority zones: CGSRI vs population\n(all {n_pz} priority cells)', fontsize=9)
    total_pop = df_pz[pz_pop].sum()
    n_vh = len(df_pz[df_pz.get('cgsri_class', pd.Series()) == 'Very high']) if 'cgsri_class' in df_pz.columns else \
           len(gdf_cs[gdf_cs['cgsri_class'] == 'Very high'])
    ax.text(0.03, 0.97,
            f'Priority zone summary\n'
            f'N cells shown: {n_pz}\n'
            f'Total population: {total_pop/1e6:.1f}M\n'
            f'N Very High cells: {n_vh}\n'
            f'CGSRI range: {df_pz[pz_cgsri].min():.3f}–{df_pz[pz_cgsri].max():.3f}',
            transform=ax.transAxes, ha='left', va='top', fontsize=7.5,
            bbox=dict(boxstyle='round', fc='#E8F4F8', alpha=0.9))
else:
    ax.text(0.5, 0.5, 'Check cgsri_priority_zones.csv',
            ha='center', va='center', transform=ax.transAxes, color='gray')
    ax.set_title('(b) Priority zones', fontsize=9)

plt.tight_layout()
plt.savefig(FIGS / 'fig10_district_ranking.png', dpi=DPI)
plt.close()
print(f'Saved fig10_district_ranking.png  (missing from grid: {missing_regions})')

## Output Check

In [ ]:
figs_expected = [
    'fig01_study_area.png',
    'fig02_gwsa_timeseries.png',
    'fig03_gwsa_trend_drought_map.png',
    'fig04_climate_groundwater.png',
    'fig05_urban_growth.png',
    'fig06_ugpi_map.png',
    'fig07_subsidence_s1.png',
    'fig08_bivariate_gw_subsidence.png',
    'fig09_cgsri_risk_map.png',
    'fig10_district_ranking.png',
]
print('\n=== Notebook 12 Output Check ===')
all_ok = True
for fn in figs_expected:
    p = FIGS / fn
    status = '✓' if p.exists() else '✗ MISSING'
    sz     = f'({p.stat().st_size/1024:.0f} KB)' if p.exists() else ''
    print(f'  {status}  {fn}  {sz}')
    if not p.exists(): all_ok = False
print()
if all_ok:
    print('All 10 publication figures saved at 300 dpi.')
    print('Note: Fig 04 will improve once NB08 (TerraClimate PDSI) is executed.')
    print('Next: Notebook 13 (Streamlit dashboard) or policy brief.')
else:
    print('Some figures missing — check cell errors above.')